# Phase 6 — Self-Supervised Masked Time-Series Reconstruction

**Objective**: Test whether a Temporal Transformer can learn useful representations from sensor time-series data *without anomaly labels* (self-supervised pretraining), and then use those representations for anomaly classification.

**Data leakage policy**:
- SSL pretraining uses **training data only** (normal windows, no labels)
- Validation data is used only for early stopping and threshold selection
- Test set is untouched until final evaluation
- Anomaly labels are used **only** during supervised fine-tuning

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path

from config import RESULTS_DIR, FIGURES_DIR, MODELS_DIR, SEQUENCE_LENGTH

SSL_RES  = RESULTS_DIR / 'ssl'
SSL_FIGS = FIGURES_DIR / 'ssl'
SSL_MOD  = MODELS_DIR  / 'ssl'

print('Results dir:', SSL_RES)
print('Figures dir:', SSL_FIGS)

## 2. Existing Data Pipeline

In [ ]:
from data_loader import load_raw
from preprocessing import run_preprocessing
from datasets import WaterSensorDataset, MaskedSensorDataset

df = load_raw()
train_df, val_df, test_df, scaler, feat_cols = run_preprocessing(df, save=False)

STRIDE = 12
train_sup = WaterSensorDataset(train_df, feat_cols, label_strategy='last', stride=STRIDE)
val_sup   = WaterSensorDataset(val_df,   feat_cols, label_strategy='last', stride=STRIDE)
test_sup  = WaterSensorDataset(test_df,  feat_cols, label_strategy='last', stride=STRIDE)

print(f'Features ({len(feat_cols)}): {feat_cols}')
print(f'Train windows: {len(train_sup):,}  anomaly={train_sup.labels.mean()*100:.1f}%')
print(f'Val   windows: {len(val_sup):,}')
print(f'Test  windows: {len(test_sup):,}')

## 3. Masking Strategy

In [ ]:
# MaskedSensorDataset already implemented in src/datasets.py
# Uses only normal (label=0) windows for SSL pretraining
train_ssl_ds = MaskedSensorDataset(train_df, feat_cols, stride=STRIDE, mask_ratio=0.15)
val_ssl_ds   = MaskedSensorDataset(val_df,   feat_cols, stride=STRIDE, mask_ratio=0.15)

print(f'Train SSL windows (normal only): {len(train_ssl_ds):,}')
print(f'Val   SSL windows (normal only): {len(val_ssl_ds):,}')

# Inspect one sample
masked_x, original_x, mask = train_ssl_ds[0]
print(f'\nWindow shape   : {original_x.shape}  (seq_len={SEQUENCE_LENGTH}, n_features={len(feat_cols)})')
print(f'Mask shape     : {mask.shape}')
print(f'Actual mask %  : {mask.mean()*100:.1f}%  (target=15%)')
print(f'Masked positions: {int(mask.sum())} of {mask.numel()}')

## 4. SSL Transformer Architecture

In [ ]:
from self_supervised import SSLTransformer

ssl_cfg = json.load(open(SSL_MOD / 'ssl_config.json'))
print('SSL config:')
for k, v in ssl_cfg.items():
    print(f'  {k}: {v}')

model = SSLTransformer(
    n_features=ssl_cfg['n_features'],
    d_model=ssl_cfg['d_model'],
    nhead=ssl_cfg['nhead'],
    num_layers=ssl_cfg['num_layers'],
    dim_feedforward=ssl_cfg['dim_feedforward'],
    dropout=ssl_cfg['dropout'],
)
print(f'\nTotal parameters: {model.n_params:,}')
print('\nArchitecture:')
print(model)

## 5. Pretraining Results

In [ ]:
from IPython.display import Image, display

print(f"Best pretrain epoch : {ssl_cfg['best_pretrain_epoch']}")
print(f"Best val recon loss : {ssl_cfg['best_val_recon_loss']}")
print(f"Pretrain time       : {ssl_cfg['total_pretrain_time_s']:.1f}s")

display(Image(str(SSL_FIGS / 'ssl_training_loss.png')))

## 6. Reconstruction Quality

In [ ]:
sensor_err = pd.read_csv(SSL_RES / 'sensor_reconstruction_errors.csv')
print('Per-sensor masked MSE (sorted hardest → easiest):')
print(sensor_err.to_string(index=False))

display(Image(str(SSL_FIGS / 'reconstruction_examples.png')))
display(Image(str(SSL_FIGS / 'per_sensor_recon_error.png')))

**Observations**:
- n215 (pressure) has the highest reconstruction error — it shows the most abrupt changes in the dataset
- T1 (tank level) is the second hardest — slow drift patterns require longer context
- Temporal features (sin/cos hour, dow) are easiest to reconstruct — they are deterministic
- Reconstruction loss decreases steadily across all 10 epochs with no sign of overfitting

## 7. Fine-tuning Results

In [ ]:
display(Image(str(SSL_FIGS / 'finetuning_curves.png')))

## 8. Standard Transformer vs SSL Transformer

In [ ]:
comp = pd.read_csv(SSL_RES / 'transformer_vs_ssl.csv')
display(comp[['model','f1','precision','recall','roc_auc','pr_auc','fpr','fnr']])
display(Image(str(SSL_FIGS / 'transformer_vs_ssl_comparison.png')))

## 9. Masking-Ratio Ablation

In [ ]:
abl = pd.read_csv(SSL_RES / 'masking_ratio_ablation.csv')
print('Masking-ratio ablation results:')
display(abl)
display(Image(str(SSL_FIGS / 'masking_ratio_ablation.png')))

## 10. Error Analysis

In [ ]:
err = pd.read_csv(SSL_RES / 'error_analysis.csv')
print('Error comparison (Standard vs SSL):')
print('Standard Transformer — FP=29  FN=18')
print('SSL Transformer      — FP=34  FN=8')
print()
print('Errors unique to each model:')
print(err['error_type'].value_counts().to_string())
display(err)

## 11. Findings

**SSL pretraining**:
- Reconstruction loss decreased consistently over 10 epochs (0.866 → 0.155 train, 0.558 → 0.119 val)
- No overfitting observed during pretraining — val loss tracked train loss closely
- Hardest sensors to reconstruct: n215 (pressure, abrupt changes), T1 (tank level, slow drift)
- Easiest: cyclical temporal features (sin/cos hour, dow) — deterministic patterns

**Comparison (test set)**:

| Metric | Standard Transformer | SSL Transformer | Change |
|--------|---------------------|-----------------|--------|
| F1 | 0.9221 | 0.9320 | +0.0099 |
| ROC-AUC | 0.9934 | 0.9919 | -0.0015 |
| PR-AUC | 0.9769 | 0.9851 | +0.0082 |
| FPR | 0.0249 | 0.0292 | +0.0043 |
| FNR | 0.0608 | 0.0270 | -0.0338 |

SSL pretraining improved **recall** (0.9392 → 0.9730) and **FNR** substantially (0.0608 → 0.0270), meaning fewer missed anomalies. This is the most safety-critical metric for water system monitoring. The trade-off is a slight increase in FPR (more false alarms).

**Masking-ratio ablation**:
- mask_ratio=0.10: F1=0.9633, ROC-AUC=0.9975 (best ROC-AUC)
- mask_ratio=0.15: F1=0.9666, ROC-AUC=0.9971 (best F1)
- mask_ratio=0.30: F1=0.9568, ROC-AUC=0.9942 (worst — too much masking degrades representations)

Note: ablation used 5 pretrain epochs vs 10 for the main model, so absolute values differ.

## 12. Limitations

- CPU-only training: only 5 fine-tuning epochs and 10 pretraining epochs. More epochs may further improve results.
- stride=12 reduces the training set to 5,829 windows — a larger dataset would benefit SSL more.
- Fine-tuning val F1=0.3729 at threshold=0.5 is low, but threshold selection on val (0.0297) recovers strong test performance — consistent with Phase 5 behaviour.
- The dataset is synthetic (generated from BattLeDIM SCADA statistics). Results may differ on real sensor data.
- SSL pretraining uses normal-only windows. Including anomalous windows with a different masking strategy could be explored.